In [1]:
import pandas as pd
import numpy as np
import time

CSV_PATH = r"D:\UniStuff\BA\BA-Adrian-Preisler\BE\data\DataSets\titanic.csv"

df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
display(df.head(10))

Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [2]:
TARGET_VARIABLE = "Survived"

if TARGET_VARIABLE not in df.columns:
    raise ValueError(f"Target '{TARGET_VARIABLE}' not found in dataframe.")

X = df.drop(columns=[TARGET_VARIABLE])
y = df[TARGET_VARIABLE]

non_numeric_cols = list(X.select_dtypes(exclude=["number"]).columns)
print("Entfernte nicht-numerische Spalten:", non_numeric_cols)

X = X.select_dtypes(include=["number"])

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Klassen / Werte:")
print(y.value_counts(dropna=False))

Entfernte nicht-numerische Spalten: ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']
X shape: (891, 6)
y shape: (891,)
Klassen / Werte:
Survived
0    549
1    342
Name: count, dtype: int64


In [3]:
def _infer_task_type(y, max_classes_for_classification: int = 20) -> str:
    y_values = y.dropna()

    # object / bool / category => classification
    if str(y_values.dtype) in ("object", "bool", "category"):
        return "classification"

    n_unique = int(y_values.nunique(dropna=True))
    n = int(len(y_values))

    # int-like and few unique -> classification
    if np.issubdtype(y_values.dtype, np.integer) and n_unique <= max_classes_for_classification:
        return "classification"

    # few unique relative to dataset size -> classification
    if n_unique <= max_classes_for_classification and (n_unique / max(1, n)) < 0.05:
        return "classification"

    return "regression"


def _can_stratify(y) -> bool:
    try:
        vc = y.value_counts(dropna=False)
        return int(vc.min()) >= 2
    except Exception:
        vals, counts = np.unique(np.asarray(y), return_counts=True)
        return int(counts.min()) >= 2


task_type = _infer_task_type(y)
print("Erkannter Task-Type:", task_type)
print("Stratify möglich:", _can_stratify(y))

Erkannter Task-Type: classification
Stratify möglich: True


In [4]:
from sklearn.model_selection import train_test_split

strat = y if (task_type == "classification" and _can_stratify(y)) else None

X_train_full, X_test_data, y_train_full, y_test_data = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=1337,
    stratify=strat,
)

print("Train Data:", X_train_full.shape, "from", X.shape)
print("Test Data:", X_test_data.shape, "from", X.shape)

Train Data: (712, 6) from (891, 6)
Test Data: (179, 6) from (891, 6)


In [5]:
strat_train = y_train_full if (task_type == "classification" and _can_stratify(y_train_full)) else None

X_train_data, X_val_data, y_train_data, y_val_data = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=1337,
    stratify=strat_train,
)

print("Train Data:", X_train_data.shape, "from", X_train_full.shape)
print("Validation Data:", X_val_data.shape, "from", X_train_full.shape)

Train Data: (569, 6) from (712, 6)
Validation Data: (143, 6) from (712, 6)


In [6]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import (
    StratifiedKFold,
    KFold,
    cross_val_score,
    ParameterGrid,
    ParameterSampler,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    fbeta_score,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)
from scipy.stats import randint

In [7]:
# STEUER-VARIABLEN
MODE = "none"              # "none" | "random" | "grid"
SCORING = "accuracy"       # wird unten automatisch angepasst
N_SPLITS = 5
N_ITER = 25
USE_CROSS_VALIDATION = False
TIME_LIMIT_S = 60
SEED = 1337

In [8]:
if task_type == "classification":
    main_scoring = "accuracy"
    SCORING = "accuracy"

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    ModelCls = RandomForestClassifier
    base_model_kwargs = dict(
        random_state=SEED,
        n_jobs=-1,
        max_features="sqrt",
    )

    param_grid = {
        "n_estimators": [300, 700, 1200],
        "max_depth": [None, 8, 16],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "criterion": ["entropy"],
    }

    param_dist = {
        "n_estimators": randint(300, 2001),
        "max_depth": [None] + list(range(3, 31)),
        "min_samples_split": randint(2, 41),
        "min_samples_leaf": randint(1, 21),
        "criterion": ["entropy"],
    }

    sklearn_selection_scoring = "accuracy"

else:
    main_scoring = "mae"
    SCORING = "mae"

    cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    ModelCls = RandomForestRegressor
    base_model_kwargs = dict(
        random_state=SEED,
        n_jobs=-1,
        max_features=1.0,
    )

    param_grid = {
        "n_estimators": [300, 700, 1200],
        "max_depth": [None, 8, 16],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "criterion": ["squared_error"],
    }

    param_dist = {
        "n_estimators": randint(300, 2001),
        "max_depth": [None] + list(range(3, 31)),
        "min_samples_split": randint(2, 41),
        "min_samples_leaf": randint(1, 21),
        "criterion": ["squared_error"],
    }

    sklearn_selection_scoring = "neg_mean_absolute_error"

print("MODE:", MODE)
print("SCORING:", SCORING)
print("USE_CROSS_VALIDATION:", USE_CROSS_VALIDATION)

MODE: none
SCORING: accuracy
USE_CROSS_VALIDATION: False


In [9]:
def _selection_score(y_true, y_pred):
    if task_type == "classification":
        return float(accuracy_score(y_true, y_pred))
    return float(-mean_absolute_error(y_true, y_pred))


t0 = time.time()

def time_left():
    return (time.time() - t0) < max(1, int(TIME_LIMIT_S))

In [10]:
best_val_score = -np.inf
best_params = None
best_model = None

if MODE == "none":
    if task_type == "classification":
        fixed_params = {
            "n_estimators": 500,
            "min_samples_leaf": 2,
            "max_depth": None,
            "criterion": "entropy",
            "min_samples_split": 2,
        }
    else:
        fixed_params = {
            "n_estimators": 500,
            "min_samples_leaf": 2,
            "max_depth": None,
            "criterion": "squared_error",
            "min_samples_split": 2,
        }

    model = ModelCls(**fixed_params, **base_model_kwargs)

    if USE_CROSS_VALIDATION:
        cv_scores = cross_val_score(
            model,
            X_train_full,
            y_train_full,
            cv=cv,
            scoring=sklearn_selection_scoring,
            n_jobs=-1,
        )
        best_val_score = float(np.mean(cv_scores))
        model.fit(X_train_full, y_train_full)
    else:
        model.fit(X_train_data, y_train_data)
        y_val_pred = model.predict(X_val_data)
        best_val_score = _selection_score(y_val_data, y_val_pred)

    best_model = model
    best_params = fixed_params

elif MODE == "grid":
    for params in ParameterGrid(param_grid):
        if not time_left():
            break

        m = ModelCls(**params, **base_model_kwargs)

        if USE_CROSS_VALIDATION:
            cv_scores = cross_val_score(
                m,
                X_train_full,
                y_train_full,
                cv=cv,
                scoring=sklearn_selection_scoring,
                n_jobs=-1,
            )
            val_score = float(np.mean(cv_scores))
        else:
            m.fit(X_train_data, y_train_data)
            y_val_pred = m.predict(X_val_data)
            val_score = _selection_score(y_val_data, y_val_pred)

        if best_params is None or val_score > best_val_score:
            best_val_score = val_score
            best_params = params

    if best_params is None:
        raise RuntimeError("Grid Search did not evaluate any configuration (time limit too small).")

    best_model = ModelCls(**best_params, **base_model_kwargs)
    if USE_CROSS_VALIDATION:
        best_model.fit(X_train_full, y_train_full)
    else:
        best_model.fit(X_train_data, y_train_data)

elif MODE == "random":
    sampler = ParameterSampler(param_dist, n_iter=N_ITER, random_state=SEED)

    for params in sampler:
        if not time_left():
            break

        m = ModelCls(**params, **base_model_kwargs)

        if USE_CROSS_VALIDATION:
            cv_scores = cross_val_score(
                m,
                X_train_full,
                y_train_full,
                cv=cv,
                scoring=sklearn_selection_scoring,
                n_jobs=-1,
            )
            val_score = float(np.mean(cv_scores))
        else:
            m.fit(X_train_data, y_train_data)
            y_val_pred = m.predict(X_val_data)
            val_score = _selection_score(y_val_data, y_val_pred)

        if best_params is None or val_score > best_val_score:
            best_val_score = val_score
            best_params = params

    if best_params is None:
        raise RuntimeError("Random Search did not evaluate any configuration (time limit too small).")

    best_model = ModelCls(**best_params, **base_model_kwargs)
    if USE_CROSS_VALIDATION:
        best_model.fit(X_train_full, y_train_full)
    else:
        best_model.fit(X_train_data, y_train_data)

else:
    raise ValueError('MODE must be "none", "grid", or "random"')

In [11]:
y_pred_test = best_model.predict(X_test_data)

if task_type == "classification":
    acc = float(accuracy_score(y_test_data, y_pred_test))
    f1m = float(f1_score(y_test_data, y_pred_test, average="macro"))
    f2m = float(fbeta_score(y_test_data, y_pred_test, beta=2, average="micro"))

    best_val_score_out = float(best_val_score)

    print(f"RANDOM FOREST ({MODE.upper()})")
    print(f"Task-Type: {task_type}")
    print(f"Best Val ({SCORING}): {best_val_score_out:.4f}")
    print(f"Test Acc: {acc:.4f} | Test F1(macro): {f1m:.4f} | Test F2(micro): {f2m:.4f}")

else:
    mae = float(mean_absolute_error(y_test_data, y_pred_test))
    rmse = float(mean_squared_error(y_test_data, y_pred_test, squared=False))
    r2 = float(r2_score(y_test_data, y_pred_test))

    best_val_score_out = float(-best_val_score)

    print(f"RANDOM FOREST ({MODE.upper()})")
    print(f"Task-Type: {task_type}")
    print(f"Best Val ({SCORING}): {best_val_score_out:.4f}")
    print(f"Test MAE: {mae:.4f} | Test RMSE: {rmse:.4f} | Test R2: {r2:.4f}")

RANDOM FOREST (NONE)
Task-Type: classification
Best Val (accuracy): 0.7832
Test Acc: 0.7486 | Test F1(macro): 0.7273 | Test F2(micro): 0.7486


In [12]:
p = best_model.get_params()
used = {
    k: p.get(k) for k in [
        "n_estimators",
        "min_samples_leaf",
        "max_depth",
        "criterion",
        "min_samples_split",
        "max_features",
    ]
}

print("Verwendete Hyperparameter:")
print(used)

Verwendete Hyperparameter:
{'n_estimators': 500, 'min_samples_leaf': 2, 'max_depth': None, 'criterion': 'entropy', 'min_samples_split': 2, 'max_features': 'sqrt'}


In [13]:
import time
import numpy as np

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    KFold,
    cross_val_score,
    ParameterGrid,
    ParameterSampler,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    fbeta_score,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)
from scipy.stats import randint


def _infer_task_type(y, max_classes_for_classification: int = 20) -> str:
    y_values = y.dropna()

    # object / bool / category => classification
    if str(y_values.dtype) in ("object", "bool", "category"):
        return "classification"

    n_unique = int(y_values.nunique(dropna=True))
    n = int(len(y_values))

    # int-like and few unique -> classification
    if np.issubdtype(y_values.dtype, np.integer) and n_unique <= max_classes_for_classification:
        return "classification"

    # few unique relative to dataset size -> classification
    if n_unique <= max_classes_for_classification and (n_unique / max(1, n)) < 0.05:
        return "classification"

    return "regression"


def _can_stratify(y) -> bool:
    try:
        vc = y.value_counts(dropna=False)
        return int(vc.min()) >= 2
    except Exception:
        vals, counts = np.unique(np.asarray(y), return_counts=True)
        return int(counts.min()) >= 2


def train_random_forest(
    df_encoded,
    target_variable: str,
    MODE: str = "none",
    SCORING: str = "accuracy",
    N_SPLITS: int = 5,
    N_ITER: int = 25,
    use_cross_validation: bool = False,
    time_limit_s: int = 60,
    seed: int = 1337,
    task_type: str | None = None,
):

    TARGET_VARIABLE = target_variable
    if TARGET_VARIABLE not in df_encoded.columns:
        raise ValueError(f"Target '{TARGET_VARIABLE}' not in df columns.")

    X = df_encoded.drop(columns=[TARGET_VARIABLE])
    y = df_encoded[TARGET_VARIABLE]

    non_numeric_cols = list(X.select_dtypes(exclude=["number"]).columns)
    if non_numeric_cols:
        raise ValueError(
            f"df_encoded must be fully numeric after preprocessing. Non-numeric columns: {non_numeric_cols[:20]}"
            + (" ..." if len(non_numeric_cols) > 20 else "")
        )

    if task_type is None:
        task_type = _infer_task_type(y)

    if task_type not in ("classification", "regression"):
        raise ValueError('task_type must be "classification" or "regression" (or None for auto).')

    if task_type == "classification":
        main_scoring = "accuracy"
        if SCORING not in ("accuracy"):
            SCORING = "accuracy"
    else:
        main_scoring = "mae"
        SCORING = "mae"

    strat = None
    if task_type == "classification" and _can_stratify(y):
        strat = y

    X_train_full, X_test_data, y_train_full, y_test_data = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=seed,
        stratify=strat,
    )

    strat_train = None
    if task_type == "classification" and _can_stratify(y_train_full):
        strat_train = y_train_full

    X_train_data, X_val_data, y_train_data, y_val_data = train_test_split(
        X_train_full,
        y_train_full,
        test_size=0.2,
        random_state=seed,
        stratify=strat_train,
    )

    if task_type == "classification":
        cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    else:
        cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

    base_model_kwargs = dict(random_state=seed, n_jobs=-1)

    if task_type == "classification":
        ModelCls = RandomForestClassifier
        base_model_kwargs["max_features"] = "sqrt"
        param_grid = {
            "n_estimators": [300, 700, 1200],
            "max_depth": [None, 8, 16],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4],
            "criterion": ["entropy"],
        }
        param_dist = {
            "n_estimators": randint(300, 2001),
            "max_depth": [None] + list(range(3, 31)),
            "min_samples_split": randint(2, 41),
            "min_samples_leaf": randint(1, 21),
            "criterion": ["entropy"],
        }
    else:
        ModelCls = RandomForestRegressor
        base_model_kwargs["max_features"] = 1.0
        param_grid = {
            "n_estimators": [300, 700, 1200],
            "max_depth": [None, 8, 16],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4],
            "criterion": ["squared_error"],
        }
        param_dist = {
            "n_estimators": randint(300, 2001),
            "max_depth": [None] + list(range(3, 31)),
            "min_samples_split": randint(2, 41),
            "min_samples_leaf": randint(1, 21),
            "criterion": ["squared_error"],
        }

    def _selection_score(y_true, y_pred):
        if task_type == "classification":
            return float(accuracy_score(y_true, y_pred))
        return float(-mean_absolute_error(y_true, y_pred))

    if task_type == "classification":
        sklearn_selection_scoring = "accuracy"
    else:
        sklearn_selection_scoring = "neg_mean_absolute_error"

    t0 = time.time()

    def time_left():
        return (time.time() - t0) < max(1, int(time_limit_s))

    best_val_score = -np.inf
    best_params = None
    best_model = None

    if MODE == "none":
        if task_type == "classification":
            fixed_params = {
                "n_estimators": 500,
                "min_samples_leaf": 2,
                "max_depth": None,
                "criterion": "entropy",
                "min_samples_split": 2,
            }
        else:
            fixed_params = {
                "n_estimators": 500,
                "min_samples_leaf": 2,
                "max_depth": None,
                "criterion": "squared_error",
                "min_samples_split": 2,
            }

        model = ModelCls(**fixed_params, **base_model_kwargs)

        if use_cross_validation:
            cv_scores = cross_val_score(
                model,
                X_train_full,
                y_train_full,
                cv=cv,
                scoring=sklearn_selection_scoring,
                n_jobs=-1,
            )
            best_val_score = float(np.mean(cv_scores))
            model.fit(X_train_full, y_train_full)
        else:
            model.fit(X_train_data, y_train_data)
            y_val_pred = model.predict(X_val_data)
            best_val_score = _selection_score(y_val_data, y_val_pred)

        best_model = model
        best_params = fixed_params

    elif MODE == "grid":
        for params in ParameterGrid(param_grid):
            if not time_left():
                break

            m = ModelCls(**params, **base_model_kwargs)

            if use_cross_validation:
                cv_scores = cross_val_score(
                    m,
                    X_train_full,
                    y_train_full,
                    cv=cv,
                    scoring=sklearn_selection_scoring,
                    n_jobs=-1,
                )
                val_score = float(np.mean(cv_scores))
            else:
                m.fit(X_train_data, y_train_data)
                y_val_pred = m.predict(X_val_data)
                val_score = _selection_score(y_val_data, y_val_pred)

            if best_params is None or val_score > best_val_score:
                best_val_score = val_score
                best_params = params

        if best_params is None:
            raise RuntimeError("Grid Search did not evaluate any configuration (time limit too small).")

        best_model = ModelCls(**best_params, **base_model_kwargs)
        if use_cross_validation:
            best_model.fit(X_train_full, y_train_full)
        else:
            best_model.fit(X_train_data, y_train_data)

    elif MODE == "random":
        sampler = ParameterSampler(param_dist, n_iter=N_ITER, random_state=seed)

        for params in sampler:
            if not time_left():
                break

            m = ModelCls(**params, **base_model_kwargs)

            if use_cross_validation:
                cv_scores = cross_val_score(
                    m,
                    X_train_full,
                    y_train_full,
                    cv=cv,
                    scoring=sklearn_selection_scoring,
                    n_jobs=-1,
                )
                val_score = float(np.mean(cv_scores))
            else:
                m.fit(X_train_data, y_train_data)
                y_val_pred = m.predict(X_val_data)
                val_score = _selection_score(y_val_data, y_val_pred)

            if best_params is None or val_score > best_val_score:
                best_val_score = val_score
                best_params = params

        if best_params is None:
            raise RuntimeError("Random Search did not evaluate any configuration (time limit too small).")

        best_model = ModelCls(**best_params, **base_model_kwargs)
        if use_cross_validation:
            best_model.fit(X_train_full, y_train_full)
        else:
            best_model.fit(X_train_data, y_train_data)

    else:
        raise ValueError('MODE must be "none", "grid", or "random"')

    y_pred_test = best_model.predict(X_test_data)

    if task_type == "classification":
        acc = float(accuracy_score(y_test_data, y_pred_test))
        f1m = float(f1_score(y_test_data, y_pred_test, average="macro"))
        f2m = float(fbeta_score(y_test_data, y_pred_test, beta=2, average="micro"))
        metrics = {
            "test_accuracy": acc,
            "test_f1_macro": f1m,
            "test_f2_micro": f2m,
        }

        best_val_score_out = float(best_val_score)
    else:
        mae = float(mean_absolute_error(y_test_data, y_pred_test))
        rmse = float(mean_squared_error(y_test_data, y_pred_test, squared=False))
        r2 = float(r2_score(y_test_data, y_pred_test))
        metrics = {
            "test_mae": mae,
            "test_rmse": rmse,
            "test_r2": r2,
        }

        best_val_score_out = float(-best_val_score)

    p = best_model.get_params()
    used = {k: p.get(k) for k in [
        "n_estimators",
        "min_samples_leaf",
        "max_depth",
        "criterion",
        "min_samples_split",
        "max_features",
    ]}

    return {
        "model": "Random Forest",
        "model_key": "rf",
        "task_type": task_type,
        "search_mode": MODE,
        "cross_validation": bool(use_cross_validation),
        "main_scoring": ("accuracy" if task_type == "classification" else "mae"),
        "time_limit_s": int(time_limit_s),
        "seed": int(seed),
        "hyperparameters": used,
        "best_val_score": best_val_score_out,
        "metrics": metrics,
        "_model": best_model,
    }